# Legacy decoder UI - quarantined

This pre-foundation UI is retained only as non-executable provenance. It references superseded encoder/decoder paths and must not be used for Stage 2. A new inference UI may be implemented after Stage 3 establishes approved five-fold decoder checkpoints.

# Decoder UI - interactive 3D reconstruction tester

A small **Gradio** app to test the trained decoders. Pick a **fold**, a **regime** (`frozen` /
`finetuned`), a **model** (**U-Net** or **V-Net**) and a **checkpoint epoch**, choose a built-in
test case *or upload your own AP + LAT images* (e.g. real **Regen** clinical X-rays), and view the
reconstructed 3D bone surface plus mid-slices. When a ground-truth CT exists for the chosen case it
also reports Dice/IoU.

This is how you **revisit any saved epoch** of any fold/regime/model run from `03_decoder_pipeline.ipynb`,
and how doctors can **qualitatively review** Regen reconstructions (which have no 3D ground truth).

### Setup

This app needs **gradio** (not required for training). Install it once, then restart the kernel:

### Shared encoder (copied verbatim from `01_encoder_pipeline.ipynb`)

The encoder is the part both decoders share, so the comparison is fair: **only the decoder
changes**. The code below is copied verbatim from `01_encoder_pipeline.ipynb` so this notebook is
self-contained Ã¢â‚¬â€ **do not edit it here**. (When `01_encoder_pipeline.ipynb` is later converted to a
`.py` module, replace this cell with a simple `import`.)

What it does, in plain terms:
1. A **ConvNeXtV2** backbone turns each X-ray (AP and LAT) into 4 feature maps at increasing depth.
2. **Hybrid bi-planar fusion** merges the two views: cheap convolution at fine scales (keeps local
   fracture detail), cross-attention at coarse scales (aligns global knee shape).
3. A **2D->3D lift** stacks each fused map into a small 3D feature volume (depth = `LIFT_DEPTH`).

Output: a list of 4 multi-scale 3D feature tensors with channels `[64, 128, 256, 512]` Ã¢â‚¬â€ this is
the *contract* the decoder consumes.

### Inference helpers

`load_model_from_ckpt` rebuilds the exact architecture recorded in the checkpoint's config
(model type, resolution, lift depth) and loads its weights. `to_input` normalises any image to the
encoder's expected `3x256x256` tensor. `volume_to_obj` turns the predicted occupancy volume into a
surface mesh with marching cubes and writes a small `.obj` for the 3D viewer.

### The app

Press **Reconstruct** to run inference. The 3D viewer shows the predicted bone surface; the plot
shows the input AP view and two prediction slices; the info box reports the checkpoint used and, for
built-in cases with a CT, the Dice/IoU.

## Notes

- The dropdowns walk `models/decoders/fold{FOLD}/{regime}/{model}/` and let you drill down
  **fold -> regime -> model -> epoch**. Run `03_decoder_pipeline.ipynb` (any fold/regime/model) first so
  at least one checkpoint exists.
- Each decoder checkpoint stores the **full** model (the frozen `front_end_fold{FOLD}.pth` front-end
  **plus** the decoder), so loading a single `.pth` here restores the exact front-end used in
  training Ã¢â‚¬â€ there is no separate front-end file to select.
- For **Regen** clinical X-rays, choose **upload** and provide the AP and LAT images; there is no
  ground truth, so you get the mesh + slices for qualitative review (no Dice/IoU).
- `demo.launch(share=True)` gives a temporary public link if you need to show it to a supervisor.